# Download Dataset Mask Gigi dari Roboflow

Mengunduh anotasi segmentasi gigi (mask) untuk 3 project OMNI (train/val/test) ke
`data/masks/{train,val,test}/`.

**Cara pakai:**
1. Buka `roboflow_config.json` (di folder yang sama), isi `api_key`, cek project/version/format.
2. Jalankan semua cell di bawah.

Format `coco-segmentation` memberi anotasi polygon (yang kita rasterisasi jadi mask nanti).
Karena label satu kelas "tooth", ini cukup untuk background masking.

In [12]:
import os, json, sys, subprocess

try:    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError: _HERE = os.getcwd()
PROJECT_ROOT = os.path.dirname(_HERE) if os.path.basename(_HERE) == 'notebooks' else _HERE

# cari file config
CFG_PATH = None
for cand in [os.path.join(_HERE, 'roboflow_config.json'),
             os.path.join(PROJECT_ROOT, 'notebooks', 'roboflow_config.json'),
             os.path.join(PROJECT_ROOT, 'roboflow_config.json')]:
    if os.path.exists(cand): CFG_PATH = cand; break
assert CFG_PATH, 'roboflow_config.json tidak ditemukan di folder notebooks/.'
cfg = json.load(open(CFG_PATH))
assert cfg.get('api_key') and cfg['api_key'] != 'PASTE_API_KEY_HERE', \
    f'Isi api_key di {CFG_PATH} lebih dulu.'
print('Config :', os.path.relpath(CFG_PATH, PROJECT_ROOT))
print('Workspace:', cfg['workspace'], '| projects:', list(cfg['projects']))

Config : notebooks/roboflow_config.json
Workspace: ana-rusdiati | projects: ['train', 'val', 'test']


In [13]:
# pasang roboflow bila belum ada
try:
    import roboflow
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])
    import roboflow
import shutil
from roboflow import Roboflow

rf = Roboflow(api_key=cfg['api_key'])
DEST = os.path.join(PROJECT_ROOT, 'data', 'masks'); os.makedirs(DEST, exist_ok=True)

hasil = {}
for split, p in cfg['projects'].items():
    loc = os.path.join(DEST, split)
    shutil.rmtree(loc, ignore_errors=True); os.makedirs(loc, exist_ok=True)  # bersihkan agar tak menumpuk
    print(f'\n=== {split}: {p["project"]} v{p["version"]} ({p["format"]}) ===')
    ds = (rf.workspace(cfg['workspace']).project(p['project'])
            .version(p['version']).download(p['format'], location=loc, overwrite=True))
    hasil[split] = ds.location
    print('  -> tersimpan di', os.path.relpath(ds.location, PROJECT_ROOT))


=== train: omni-front-teeth-train v1 (coco) ===
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /Users/rdrusdiati/IOTN-AC/data/masks/train in coco:: 100%|██████████| 377/377 [00:00<00:00, 7524.83it/s]

  -> tersimpan di data/masks/train

=== val: omni-front-teeth-val v2 (coco) ===
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /Users/rdrusdiati/IOTN-AC/data/masks/val in coco:: 100%|██████████| 78/78 [00:00<00:00, 3319.42it/s]

  -> tersimpan di data/masks/val

=== test: omni-front-teeth-test v1 (coco) ===
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /Users/rdrusdiati/IOTN-AC/data/masks/test in coco:: 100%|██████████| 109/109 [00:00<00:00, 5542.84it/s]

  -> tersimpan di data/masks/test


In [14]:
# verifikasi isi hasil unduhan
import glob
for split, loc in hasil.items():
    imgs = glob.glob(os.path.join(loc, '**', '*.jpg'), recursive=True) + \
           glob.glob(os.path.join(loc, '**', '*.png'), recursive=True)
    jsons = glob.glob(os.path.join(loc, '**', '_annotations*.json'), recursive=True) + \
            glob.glob(os.path.join(loc, '**', '*.json'), recursive=True)
    print(f'{split:5s}: {len(imgs):4d} gambar | anotasi: {[os.path.basename(j) for j in jsons][:3]}')
print('\nSelesai. Lanjut: cell preprocessing masking (background hitam, gigi in-place) di notebook berikutnya.')

train:  374 gambar | anotasi: ['_annotations.coco.json', '_annotations.coco.json']
val  :   75 gambar | anotasi: ['_annotations.coco.json', '_annotations.coco.json']
test :  106 gambar | anotasi: ['_annotations.coco.json', '_annotations.coco.json']

Selesai. Lanjut: cell preprocessing masking (background hitam, gigi in-place) di notebook berikutnya.


In [15]:
# --- Ulang download SATU split saja (bersihkan dulu) ---
import shutil, glob
ULANG = 'test'                       # ganti ke 'train' / 'val' bila perlu
p = cfg['projects'][ULANG]
loc = os.path.join(PROJECT_ROOT, 'data', 'masks', ULANG)
shutil.rmtree(loc, ignore_errors=True); os.makedirs(loc, exist_ok=True)
print(f'Mengunduh ulang {ULANG}: {p["project"]} v{p["version"]} ({p["format"]}) -> {loc}')
ds = (rf.workspace(cfg['workspace']).project(p['project'])
        .version(p['version']).download(p['format'], location=loc, overwrite=True))
n = len(glob.glob(os.path.join(ds.location, '**', '*.jpg'), recursive=True))
print('  selesai:', os.path.relpath(ds.location, PROJECT_ROOT), '|', n, 'gambar  (test harus 106)')

Mengunduh ulang test: omni-front-teeth-test v1 (coco) -> /Users/rdrusdiati/IOTN-AC/data/masks/test
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /Users/rdrusdiati/IOTN-AC/data/masks/test in coco:: 100%|██████████| 109/109 [00:00<00:00, 3685.77it/s]

  selesai: data/masks/test | 106 gambar  (test harus 106)


## Langkah berikutnya

Setelah unduhan selesai, kita akan:
1. Rasterisasi polygon COCO → mask biner per foto (cocokkan ke foto OMNI lewat nama file).
2. Terapkan mask: background hitam, gigi tetap di posisi (opsi crop-ketat ke bbox mask).
3. Sanity-check visual, lalu uji `teeth_masked` vs `full` lewat cross-validation (QWK OOF) di notebook 10.